### Exploration with local Qwen3.5-2B model

In [5]:
from transformers import pipeline

model = "Qwen/Qwen3.5-2B"

generator = pipeline(
    "text-generation",
     model=model,
     max_length=None,
     max_new_tokens=150, 
     do_sample=True,
     return_full_text=False
)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from app.app import load_resources

2026-04-16 22:48:44.806 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 22:48:44.807 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-16 22:48:44.808 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-16 22:48:44.808 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-16 22:48:44.809 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 22:48:44.834 
  command:

    streamlit run /Users/randalllee/miniforge3/envs/dsci-575-project/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-04-16 22:48:44.834 Thread 'MainThread': missing ScriptRunContext! This w

In [7]:
from app.app import load_resources

In [8]:
documents, bm25 = load_resources() # Create documents

In [9]:
from src.semantic import create_faiss_index

vectorstore = create_faiss_index(documents, 10000, reload_index=False)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
from langchain_huggingface import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=generator)

In [11]:
SYSTEM_PROMPT = """
    /no_think
    You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible."""

def build_prompt(query, context):
    return f"""{SYSTEM_PROMPT}

context:
{context}

question: 
{query}

Recommend ONE product using the context.
Do not add additional explanations or repeat the prompt.
Stop after the recommendation.

Return the answer exactly in this format:

Product Title:
Product ASIN:
Product Rating:
Product Review:
Reason for Recommendation: Write 2 natural sentences describing the product’s key benefits using evidence from the review and rating.

END
"""

In [12]:
def build_context(docs):
    return "\n\n".join(
        f"Product ASIN: {doc.metadata.get('asin')}\n"
        f"Product Title: {doc.metadata.get('product_title')}\n"
        f"Product Rating: {doc.metadata.get('product_rating')}\n"
        f"Product Review: {doc.metadata.get('product_review')}\n"
        for doc in docs
    )

In [13]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

format_context = RunnableLambda(build_context)
def prompt_builder(inputs):
    return build_prompt(inputs["input"], inputs["context"])

prompt = RunnableLambda(prompt_builder)


rag_chain = (
    {
        "context": retriever | format_context,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [14]:
query = "Moisturizing shampoo for thick curly hair"

response = rag_chain.invoke(query)

print(response)

</think>

Product Title: Just For Me Curl Peace Ultimate Detangling Shampoo
Product ASIN: B08MBC424Z
Product Rating: 5.0
Product Review: This product works great if you have thick curly hair.
Reason for Recommendation: This product works great for thick curly hair with a perfect 5-star rating confirming effectiveness for your hair type.


### Testing online Meta-Llama-3-8B-Instruct model

In [20]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

llm_endpoint = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="text-generation", # Keep this as text-generation for the base
    max_new_tokens=100,
    huggingfacehub_api_token=token,
    provider="auto" #"novita"
    )

llm = ChatHuggingFace(llm=llm_endpoint)

prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful Amazon shopping assistant.

    You must answer using ONLY the information in the context.

    - Recommend ONE product.
    - Do NOT use outside knowledge.
    - Do NOT include any extra text.
    - Return ONLY valid JSON.

    Context:
    {context}

    Question:
    {input}

    Return exactly in this format:

    {{
    "product_title": "",
    "product_asin": "",
    "product_rating": "",
    "product_review": "",
    "reason_for_recommendation": ""
    }}
    """
    )

rag_chain = (
    {
        "context": retriever | format_context,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke(query)

'{\n"product_title": "Just For Me Curl Peace Ultimate Detangling Shampoo",\n"product_asin": "B08MBC424Z",\n"product_rating": "5.0",\n"product_review": "This product works great if you have thick curly hair.",\n"reason_for_recommendation": "It works great for thick curly hair."\n}'

### This is how you would call it from the function

In [2]:
from src.rag_pipeline import RAGPipeline

rag = RAGPipeline()

response = rag.ask("something to keep your face moisturized all day")

print(response)

2026-04-16 22:48:50.653 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 22:48:50.654 No runtime found, using MemoryCacheStorageManager


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'product_title': 'Avon Care Rich Moisture Comforting Nourishing Cream with soybean 6.7 Fl Oz', 'product_asin': 'B07239JBJV', 'product_rating': '5.0', 'product_review': 'the face cream is good for me', 'reason_for_recommendation': 'This product is highly rated at 5.0 and has a positive review that states it is good for the user. It meets the requirement of keeping the face moisturized all day as it is a comforting and nourishing cream.'}
